[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/takeshun1984/NumeralAnalysisInGeophysics_SolidEarth/blob/main/08_FDM_2nd_homo_P-SV_CerjanZB_numba.ipynb)

### Numbaによる並列化版

このノートブックは、`08_FDM_2nd_homo_P-SV_CerjanZB.ipynb`と同じ計算（均質媒質、Cerjanの吸収境界、Zener bodyによる減衰）を、[Numba](https://numba.pydata.org/)を用いて高速化したものである。モデル設定・震源・出力・描画は08と同じで、応力と速度の更新部分だけが異なる。結果は`output/08numba`に出力するので、08の結果（`output/08`）と比較できる。

NumbaはGoogle Colabには最初からインストールされている。手元の環境で使う場合は`pip install numba`でインストールする。


In [ ]:
import numpy as np
import os

# 領域設定など
SP = np.float32
NX, NZ = 400, 400
DX, DZ = 0.4, 0.4
DT = 0.02
NTMAX = 2000

# 震源情報
I0, K0 = NX // 2, NZ // 2
T0, TS = 5.0, 0.0
MXX, MZZ, MXZ, MO = 0.0, 0.0, 1.0, 1.0
DTXZ = DT / (DX * DZ)

# --- 出力・減衰設定 ---
NTDEC, NXD, NZD = 50, 2, 2
DST = 10.0
NST = int(NX * DX / DST) - 1
NSKIP = 2
NWMAX = NTMAX // NSKIP
OUTDIR = "output/08numba"  # 08（NumPy版）と結果を比較できるよう出力先を分ける
ONAME0, WNAME0 = f"{OUTDIR}/psv.h.", f"{OUTDIR}/wav.h."

# --- 関数定義 ---
def kupper(t, ts, tr):
    if ts <= t <= ts + tr:
        return 3 * np.pi * (np.sin(np.pi * (t - ts) / tr))**3 / (4 * tr)
    else:
        return 0.0

# 配列の初期化 0~NX+1
SXX = np.zeros((NX + 2, NZ + 2), dtype=SP)
SZZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
SXZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
VX  = np.zeros((NX + 2, NZ + 2), dtype=SP)
VZ  = np.zeros((NX + 2, NZ + 2), dtype=SP)

# 均質媒質
VP, VS, RO = 6.0, 3.5, 2.3
RIG_val = RO * VS**2
LAM_val = RO * VP**2 - 2.0 * RO * VS**2

# 各格子点に物性を割り当て
RIG = np.full((NX + 2, NZ + 2), RIG_val, dtype=SP)
LAM = np.full((NX + 2, NZ + 2), LAM_val, dtype=SP)
RHO = np.full((NX + 2, NZ + 2), RO, dtype=SP)

# 波形記録用配列
VWX = np.zeros((NWMAX, NST), dtype=SP)
VWZ = np.zeros((NWMAX, NST), dtype=SP)
ISTX = np.array([int((ist+1)*DST/DX) for ist in range(NST)])
ISTZ = NZ // 2

# --- 吸収境界条件 (Cerjan 1985) パラメータ ---
BETA, NA = 0.09, 20

# --- 吸収境界係数の計算 ---
gx1, gx2 = np.ones(NX+2, dtype=SP), np.ones(NX+2, dtype=SP)
gz1, gz2 = np.ones(NZ+2, dtype=SP), np.ones(NZ+2, dtype=SP)
for i in range(1, NA + 1):
    val1 = np.exp(-BETA*(1.0-(i-0.5)/NA)**2)
    val2 = np.exp(-BETA*(1.0-i/NA)**2)
    
    gx1[i], gz1[i], gx2[i], gz2[i] = val1, val1, val2, val2

    gx1[NX-i+1], gx2[NX-i+1], gz1[NZ-i+1], gz2[NZ-i+1] = val2, val1, val2, val1

GX1, GZ1 = np.meshgrid(gx1, gz1, indexing='ij')
GX2, GZ2 = np.meshgrid(gx2, gz2, indexing='ij')

以下で、Qのモデル設定を実施している。
Zener bodyを用いるため緩和時間に関する部分と、メモリ変数の初期化を含む。

In [ ]:
F0, Q0 = 1.0 / T0, 10.0

QS, QP = np.full((NX+2, NZ+2), 1.0e6, dtype=SP), np.full((NX+2, NZ+2), 1.0e6, dtype=SP)
QS[0:NX//2-1, :] = Q0
QP[0:NX//2-1, :] = 2.0 * Q0

W0   = 2.0 * np.pi * F0
TAU  = 1.0 / W0 * (np.sqrt(1.0 + QP**(-2)) - 1.0 / QP)
TAUP_ratio = 1.0 / (W0**2 * TAU**2)
TAUS_ratio = (1.0 + W0 * TAU * QS) / (W0 * QS * TAU - (W0 * TAU)**2)
TU = -1.0 / TAU

RXX = np.zeros((NX+2, NZ+2), dtype=SP)
RZZ = np.zeros((NX+2, NZ+2), dtype=SP)
RXZ = np.zeros((NX+2, NZ+2), dtype=SP)

### Numbaによる更新関数

NumPy版（08）では配列全体の演算を1つずつ順に実行するため、Zener bodyのように式の項が多いと、配列全体をなめる演算の回数が増えて遅くなる。一方、Fortranのように格子点ごとの二重ループで書けば、1つの格子点の計算をまとめて行えるが、Pythonのループはそのままでは非常に遅い。

Numbaは、`@njit`を付けた関数をJIT（実行時）コンパイルして機械語に変換するため、Pythonで書いた二重ループをFortranやCと同程度の速さで実行できる。さらに`parallel=True`を指定し、ループを`prange`で書くと、そのループを複数のCPUコアで分担して並列に計算する。

実装上の注意点は以下の通り。

- **ループの順序**：NumPy配列（C順序）は最後の添字`k`がメモリ上で連続しているため、`k`を内側のループにする。Fortran（列優先）では最初の添字が連続なので、Fortran版（`FDM2D_4th_layered_CerjanZB.f90`）では`i`が内側になっている。
- **並列化してよい理由**：`update_stress`では各格子点が自分の位置の応力・メモリ変数だけを書き換え、速度は読むだけである。`update_velocity`はその逆である。したがって、格子点の計算順序によらず結果は変わらず、`i`方向のループを並列化できる。応力と速度の更新を別の関数に分けているのは、全ての応力を更新し終えてから速度を更新する必要があるためである。
- **式の形**：格子点ごとに計算するので、08の係数の事前計算は使わず、Fortran版と同じ理論式の形で書いている。
- **コンパイル時間**：最初に関数を呼び出したときにコンパイルが行われるため、1ステップ目だけ1秒程度余分に時間がかかる。


In [ ]:
import numba
from numba import njit, prange

print(f"Numba {numba.__version__}, 使用スレッド数: {numba.get_num_threads()}")

@njit(parallel=True, fastmath=True)
def update_stress(SXX, SZZ, SXZ, RXX, RZZ, RXZ, VX, VZ,
                  LAM, RIG, TU, TAUP_ratio, TAUS_ratio,
                  GX1, GZ1, GX2, GZ2, NX, NZ, rdx, rdz, dt):
    """メモリ変数と応力場の更新（Cerjanの吸収境界を含む）"""
    for i in prange(1, NX+1):          # i 方向を複数スレッドで分担
        for k in range(1, NZ+1):       # k 方向（メモリ上で連続）を内側に
            # 速度の空間微分
            dxvx = (VX[i  , k  ] - VX[i-1, k  ]) * rdx
            dzvz = (VZ[i  , k  ] - VZ[i  , k-1]) * rdz
            dxvz = (VZ[i+1, k  ] - VZ[i  , k  ]) * rdx
            dzvx = (VX[i  , k+1] - VX[i  , k  ]) * rdz

            lam, rig = LAM[i, k], RIG[i, k]
            tu, taup, taus = TU[i, k], TAUP_ratio[i, k], TAUS_ratio[i, k]
            den = 1.0 - tu*dt*0.5

            # メモリ変数更新 (Crank-Nicolson)
            rxxn, rzzn, rxzn = RXX[i, k], RZZ[i, k], RXZ[i, k]
            RXX[i, k] = (rxxn + tu*(rxxn*0.5 + (lam+2.0*rig)*(taup-1.0)*(dxvx+dzvz) - 2.0*rig*(taus-1.0)*dzvz)*dt) / den
            RZZ[i, k] = (rzzn + tu*(rzzn*0.5 + (lam+2.0*rig)*(taup-1.0)*(dxvx+dzvz) - 2.0*rig*(taus-1.0)*dxvx)*dt) / den
            RXZ[i, k] = (rxzn + tu*(rxzn*0.5 + rig*(taus-1.0)*(dxvz+dzvx))*dt) / den

            # 応力場更新 + 吸収境界
            SXX[i, k] = (SXX[i, k] + ((lam+2.0*rig)*taup*(dxvx+dzvz) - 2.0*rig*taus*dzvz + (RXX[i, k]+rxxn)*0.5)*dt) * GX1[i, k]*GZ1[i, k]
            SZZ[i, k] = (SZZ[i, k] + ((lam+2.0*rig)*taup*(dxvx+dzvz) - 2.0*rig*taus*dxvx + (RZZ[i, k]+rzzn)*0.5)*dt) * GX1[i, k]*GZ1[i, k]
            SXZ[i, k] = (SXZ[i, k] + (rig*taus*(dxvz+dzvx) + (RXZ[i, k]+rxzn)*0.5)*dt) * GX2[i, k]*GZ2[i, k]


@njit(parallel=True, fastmath=True)
def update_velocity(VX, VZ, SXX, SZZ, SXZ, RHO,
                    GX1, GZ1, GX2, GZ2, NX, NZ, rdx, rdz, dt):
    """速度場の更新（Cerjanの吸収境界を含む）"""
    for i in prange(1, NX+1):
        for k in range(1, NZ+1):
            # 応力の空間微分
            dxsxx = (SXX[i+1, k  ] - SXX[i  , k  ]) * rdx
            dxsxz = (SXZ[i  , k  ] - SXZ[i-1, k  ]) * rdx
            dzszz = (SZZ[i  , k+1] - SZZ[i  , k  ]) * rdz
            dzsxz = (SXZ[i  , k  ] - SXZ[i  , k-1]) * rdz

            VX[i, k] = (VX[i, k] + (dxsxx + dzsxz) / RHO[i, k] * dt) * GX2[i, k]*GZ1[i, k]
            VZ[i, k] = (VZ[i, k] + (dxsxz + dzszz) / RHO[i, k] * dt) * GX1[i, k]*GZ2[i, k]


以下でメインループを回しているが、応力と速度の更新は上で定義した`update_stress`と`update_velocity`を呼び出すだけである。震源の注入、波形の記録、スナップショットの出力は08と同じ。ループ全体の計算時間も表示するので、08（NumPy版）と比べてみること。


In [ ]:
# メインループ
import time

rDX, rDZ, DT_SP = SP(1.0/DX), SP(1.0/DZ), SP(DT)   # 単精度の定数として関数に渡す

print(f"{'Step':>5} / {NTMAX}: {'Time':>7} {'Vxmax':>12}")
os.makedirs(OUTDIR, exist_ok=True)

# 再実行しても前回の結果の続きにならないよう、波動場を初期化
for arr in (SXX, SZZ, SXZ, VX, VZ, RXX, RZZ, RXZ, VWX, VWZ):
    arr[:] = 0.0

t_start = time.perf_counter()
for it in range(1, NTMAX + 1):
    T = it * DT

    # 1. メモリ変数・応力場の更新
    update_stress(SXX, SZZ, SXZ, RXX, RZZ, RXZ, VX, VZ,
                  LAM, RIG, TU, TAUP_ratio, TAUS_ratio,
                  GX1, GZ1, GX2, GZ2, NX, NZ, rDX, rDZ, DT_SP)

    # 震源注入
    sdrop = MO * kupper(T, TS, T0) * DTXZ
    SXX[I0, K0] -= MXX * sdrop
    SZZ[I0, K0] -= MZZ * sdrop
    SXZ[I0-1:I0+1, K0-1:K0+1] -= MXZ * sdrop * 0.25

    # 2. 速度場の更新
    update_velocity(VX, VZ, SXX, SZZ, SXZ, RHO,
                    GX1, GZ1, GX2, GZ2, NX, NZ, rDX, rDZ, DT_SP)

    # 波形記録
    if it % NSKIP == 0:
        it1 = (it // NSKIP) - 1
        if it1 < NWMAX:
            VWX[it1, :] = VX[ISTX, ISTZ]
            VWZ[it1, :] = VZ[ISTX, ISTZ]

    # 進捗表示とスナップショット出力
    if it % NTDEC == 0:
        vxmax = np.abs(VX).max()
        print(f"{it:5d}/{NTMAX:5d}: T={T:6.2f}[s] vxmax={vxmax:.3e}")
        
        # 空間データ一括計算
        i_idx, k_idx = np.arange(1, NX+1, NXD), np.arange(1, NZ+1, NZD)
        ii, kk = np.meshgrid(i_idx, k_idx, indexing='ij')
        
        # 物理量の計算 (div, rot)
        d_vx_x = (VX[ii, kk] - VX[ii-1, kk]) / DX
        d_vz_z = (VZ[ii, kk] - VZ[ii, kk-1]) / DZ
        d_vx_z = (VX[ii, kk+1] - VX[ii, kk]) / DZ
        d_vz_x = (VZ[ii+1, kk] - VZ[ii, kk]) / DX
        
        out_data = np.column_stack([
            (ii*DX).ravel(), (kk*DZ).ravel(),
            VX[ii, kk].ravel(), VZ[ii, kk].ravel(),
            (d_vx_x + d_vz_z).ravel(), (d_vz_x - d_vx_z).ravel()
        ])
        np.savetxt(f"{ONAME0}{it:05d}.out", out_data, fmt='%9.3f %9.3f %12.3e %12.3e %12.3e %12.3e')

print(f"計算時間: {time.perf_counter() - t_start:.1f} s（初回はコンパイル時間を含む）")

# --- 波形データの最終保存 ---
print("波形データ保存中...")
time_axis = np.arange(1, NWMAX + 1) * DT * NSKIP
for ist in range(NST):
    wfname = f"{WNAME0}{int(ISTX[ist]*DX):04d}.dat"
    np.savetxt(wfname, np.column_stack([time_axis, VWX[:, ist], VWZ[:, ist]]), fmt='%9.3f %14.5e %14.5e')

print("完了")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# --- 設定 ---
fact = 1e4 # scale factor
dt = 0.02
nxs, nzs = NX // NXD, NZ // NZD  # 出力で間引いた格子数

time_val = 16.0
step = int(round(time_val/DT/NTDEC))*NTDEC # 出力間隔NTDECの倍数に丸める
time_val = step*DT
filename = f"{ONAME0}{step:05d}.out"

# データ読み込み
out = pd.read_csv(filename, sep=r'\s+', names=('X','Z','VX','VZ','div','rot'))
    

fig, axs = plt.subplots(2, 2, figsize=(12, 10))
plt.rcParams.update({'font.size': 10})
    
# データ整形
X = out['X'].values.reshape(nxs, nzs)-I0*DX
Z = out['Z'].values.reshape(nxs, nzs)-K0*DZ
vx_s = out['VX'].values.reshape(nxs, nzs)
vz_s = out['VZ'].values.reshape(nxs, nzs)
div_s = out['div'].values.reshape(nxs, nzs)
rot_s = out['rot'].values.reshape(nxs, nzs)
    
# 正規化
vx_s *= fact
vz_s *= fact
div_s *= fact
rot_s *= fact

# 左側：Vx
ax0 = axs[0,0]
im1 = ax0.pcolormesh(X, Z, vx_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_X$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_ylabel('Z from source [km]',fontsize=14)

# 右側：Vz
ax1 = axs[0,1]
im2 = ax1.pcolormesh(X, Z, vz_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_Z$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
fig.colorbar(im2, ax=ax1, shrink=0.5).set_ticks([])

# 左側：div v
ax0 = axs[1,0]
im1 = ax0.pcolormesh(X, Z, div_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{div}} \ \mathbf{{v}}$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_xlabel('X from source [km]',fontsize=14)
ax0.set_ylabel('Z from source [km]',fontsize=14)

# 右側：rot v
ax1 = axs[1,1]
im2 = ax1.pcolormesh(X, Z, rot_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{rot}} \ \mathbf{{v}}$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax1.set_xlabel('X from source [km]',fontsize=14)
fig.colorbar(im2, ax=ax1, shrink=0.5).set_ticks([])

for ax in axs.flatten():
    ax.set_xlim(-80, 80)
    ax.set_ylim(-80, 80)
    ax.set_aspect(1.0)
    ax.tick_params(direction="in", top=True, right=True, which='both')

plt.tight_layout()
plt.show()

In [ ]:
import glob

dir_path = OUTDIR  # ファイルがあるディレクトリ
file_pattern = 'wav.h.*.dat'
files = sorted(glob.glob(os.path.join(dir_path, file_pattern)))
dist0 = I0*DX  # 震源の x 座標

# attenuation function
A0 = 0.0016
vs_ref = 3.500

# === プロット ===
plt.rcParams.update({'font.size': 11})
plt.rcParams['xtick.direction'] = 'in' 
plt.rcParams['ytick.direction'] = 'in' 
plt.rcParams["xtick.minor.visible"] = True  
plt.rcParams["ytick.minor.visible"] = True  

plt.figure(figsize=(5, 6))

for file in files:
    data = np.loadtxt(file)
    t  = data[:, 0]
    vz = data[:, 2]
    filename = os.path.basename(file)
    dist = int(filename.split('.')[2])-dist0

    if dist > 0:
        plt.plot(t, vz, label=f"{dist} [km]")
    
A = A0/np.sqrt(t*vs_ref)

plt.plot(t, A, color='black',ls='--',label=r'$1 / \sqrt{r}$')

plt.legend()
plt.xlim(0,30)
plt.ylim(-0.0005, 0.0005)
plt.xlabel("Time [s]", fontsize=14)
plt.title(r"$Q = \infty $")
plt.tight_layout()
plt.show()

In [ ]:
# === プロット ===
plt.rcParams.update({'font.size': 11})
plt.rcParams['xtick.direction'] = 'in' 
plt.rcParams['ytick.direction'] = 'in' 
plt.rcParams["xtick.minor.visible"] = True  
plt.rcParams["ytick.minor.visible"] = True  

plt.figure(figsize=(5, 6))

for file in files:
    data = np.loadtxt(file)
    t  = data[:, 0]
    vz = data[:, 2]
    filename = os.path.basename(file)
    dist = int(filename.split('.')[2])-dist0

    if dist < 0:
        plt.plot(t, -vz, label=f"{dist} [km]")
    
A = A0/np.sqrt(t*vs_ref)

plt.plot(t, A, color='black',ls='--',label=r'$1 / \sqrt{r}$')

plt.legend()
plt.xlim(0,30)
plt.ylim(-0.0005, 0.0005)
plt.xlabel("Time [s]", fontsize=14)
plt.title("$Q = 10$")
plt.tight_layout()
plt.show()